In [ ]:
import sys
from pathlib import Path

repo_name = 'triplet-phase-lock'
repo_url = 'https://github.com/thinkthoughts/triplet-phase-lock.git'
repo_root = Path('/content') / repo_name

if not repo_root.exists():
    !git clone {repo_url}

if not (repo_root / 'src').exists():
    raise FileNotFoundError(f"Expected src/ at {repo_root / 'src'}, but it was not found.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root added to path:', repo_root)
print('src exists:', (repo_root / 'src').exists())


# 04 — cross-stage synthesis

**triplet-phase-lock**

Pipeline: expand → extend → resist

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.expand import sequence_n, sequence_n_perturbed, sequence_n_noisy, build_triplets_from_values
from src.extend import local_delta, direction_change
from src.resist import normalize_rows, cosine_scores

plt.rcParams['figure.figsize'] = (8,4)
plt.rcParams['axes.grid'] = True

In [ ]:
k = np.arange(1, 83)

families = {
    'clean': sequence_n(k),
    'weak': sequence_n_perturbed(k, 8, 6),
    'strong': sequence_n_perturbed(k, 20, 4),
    'noisy': sequence_n_noisy(k, 40),
}

trip = {n: build_triplets_from_values(v) for n, v in families.items()}

In [ ]:
plt.figure()
for n, v in families.items():
    plt.plot(k, v, label=n)
plt.title('Expand: global structure invariant')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
delta = {n: local_delta(t) for n, t in trip.items()}
drift = {n: direction_change(t) for n, t in trip.items()}

In [ ]:
plt.figure()
for n in trip:
    plt.plot(delta[n], label=n)
plt.title('Extend: magnitude')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
for n in trip:
    plt.plot(drift[n], label=n)
plt.title('Extend: directional change')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
ref = normalize_rows(trip['clean']).mean(axis=0)
ref = ref / np.linalg.norm(ref)

scores = {n: cosine_scores(t, ref) for n, t in trip.items()}
thr45 = 1 / np.sqrt(2)
thrS = 0.9995
maskS = {n: scores[n] >= thrS for n in trip}

In [ ]:
plt.figure()
for n in trip:
    plt.plot(scores[n], label=n)
plt.axhline(thr45, ls='--')
plt.axhline(thrS, ls='--')
plt.title('Resist: scores')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
for n in trip:
    plt.plot(maskS[n].astype(float), label=n)
plt.title('Resist: strict-threshold acceptance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
for n in trip:
    x = drift[n]
    y = scores[n][2:]
    plt.scatter(x, y, label=n, alpha=0.7)

plt.xlabel('directional change')
plt.ylabel('cosine score')
plt.title('Drift vs Resistance')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

- expand → invariant structure
- extend → drift emerges
- resist → strict thresholds filter drift

Key relation:

\[
\text{drift} \uparrow \Rightarrow \text{strict resistance} \downarrow
\]